# Verificación de la incorporación ENCFT

## Resultado
Base `encft`: 41 trimestres (2016-T1 a 2026-T1), 790.993 observaciones de miembros. Este cuaderno revisa la evidencia guardada por los scripts R/SQL; **no vuelve a consultar PostgreSQL** ni contiene contraseñas. La carga es una copia física por módulo y vistas derivadas.

## Contexto y método
La validación en vivo está en `verify-encft.R`; el perfil independiente de origen, en `profile-quality.R`. Se comparan pesos, claves, integridad entre módulos, series trimestrales y muestras con todas las columnas.

### Supuestos
Los factores son trimestrales; los indicadores laborales filtran edad >= 15. Los IDs originales no acreditan por sí solos un panel longitudinal. Las verificaciones de conteo Excel/Parquet no comparan cada celda del Excel. Ejecutar desde la raíz del repositorio o desde la carpeta del cuaderno.

In [1]:
from pathlib import Path
import csv, json
base = Path.cwd()
if not (base / 'qa/validation-summary.json').exists():
    base = base / 'research/mercado-laboral-dominicano/postgresql-recovery/encft'
qa = base / 'qa'
summary = json.loads((qa / 'validation-summary.json').read_text(encoding='utf-8'))
def rows(name):
    with (qa / name).open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))
print('Base:', summary['database'])
print('Validación en vivo:', summary['validated_at'])
print('Corte:', summary['data_end'])

Base: encft
Validación en vivo: 2026-08-30T02:27:52-0400
Corte: 2026-03-31


## Datos
Los originales se mantienen en `C:/Users/leona/Desktop/encuesta_trabajo_rd`. El manifiesto SHA256 identifica exactamente los procesados usados y los Excel que completaron las tasas de monedas.

In [2]:
reconciliation = rows('source-reconciliation.csv')
assert len(reconciliation) == 55
assert all(r['status'] in ('ok', 'recovered_from_excel') for r in reconciliation)
assert summary['quarters'] == 41
assert summary['rows']['miembros'] == 790993
print('Pares módulo-año conciliados:', len(reconciliation))
print('Hojas de monedas recuperadas:', sum(r['status'] == 'recovered_from_excel' for r in reconciliation))
print(json.dumps(summary['rows'], ensure_ascii=False, indent=2))

Pares módulo-año conciliados: 55
Hojas de monedas recuperadas: 4
{
  "miembros": 790993,
  "hogares": 258764,
  "viviendas": 258607,
  "diccionario": 115178,
  "tasas_monedas": 399
}


## Resultados
Consultas centrales ejecutadas por el verificador:
```sql
SELECT count(*) FROM microdatos.miembros;
SELECT count(*) FROM microdatos.miembros m
LEFT JOIN microdatos.hogares h USING (periodo, upm, vivienda, hogar)
WHERE h.fila_origen IS NULL;
SELECT count(*) FROM microdatos.miembros
WHERE edad >= 15 AND (pea <> ocupado + desocupado OR pet <> pea + inactivo);
```
Los controles completos y las tolerancias están en el script. A continuación se comprueba que el registro guardado no contiene errores.

In [3]:
checks = rows('database-quality.csv')
assert checks and all(r['estado'] != 'error' for r in checks)
print('Controles SQL:', len(checks))
for r in checks:
    if r['estado'] == 'advertencia':
        print(r['control'], ':', r['observado'], '—', r['nota'])

Controles SQL: 29
peso_miembro_hogar_distinto : 2627 — 2627 registros de 2023-T4 presentan pesos distintos entre modulos en los propios archivos fuente. Una muestra de claves/pesos se contrasto con el Excel original. Conservar y usar el factor del modulo correspondiente; no sustituirlo al unir.
tamano_hogar_distinto : 7 — 7 registros de miembros de noviembre de 2023 difieren del tamano declarado en Hogar. No corregir ni inferir composicion del hogar sin revisar estos casos.
viviendas_factor_nulo_conservado : 1 — Una observacion original conservada; no imputar factor ni usarla en sumas ponderadas sin tratamiento explicito.
id_hogar_original_no_enlaza : 73840 — 73840 registros de 2023; usar las claves estructurales verificadas, no ID_HOGAR solo.
iniciadores_no_equivalen_desocupados : 791 — Conservar ambas variables; la tasa abierta usa DESOCUPADO/PEA en personas de 15 anos y mas.
ruptura_ids_panel_2023_2024 : 1 — No hay IDs_PERSONA comunes entre 2023-T4 y 2024-T1. No inferir transiciones

In [4]:
comparison = rows('sql-versus-r.csv')
assert len(comparison) == 8
assert all(r['estado'] == 'ok' and int(r['trimestres']) == 41 for r in comparison)
roundtrip = rows('roundtrip-checks.csv')
assert len(roundtrip) == 5 and all(r['estado'] == 'ok' for r in roundtrip)
print('SQL frente a R: 8 métricas por 41 trimestres verificadas.')
for r in roundtrip:
    print(r['modulo'], ':', r['filas_comparadas'], 'filas y', r['columnas_comparadas'], 'columnas comparadas')

SQL frente a R: 8 métricas por 41 trimestres verificadas.
miembros : 4 filas y 617 columnas comparadas
hogares : 4 filas y 82 columnas comparadas
viviendas : 4 filas y 59 columnas comparadas
diccionario : 4 filas y 10 columnas comparadas
tasas_monedas : 4 filas y 19 columnas comparadas


## Límites y uso
Se usa la clave estructural para evitar los 73.840 enlaces fallidos por ID_HOGAR en 2023. En 2023-T4, 2.627 observaciones de Miembros tienen pesos distintos de Hogar; una muestra se contrastó con el Excel original. Se conserva el factor de cada módulo. Otras siete observaciones difieren en tamaño del hogar. Se conserva una vivienda sin factor y se separa DESOCUPADO_INI de DESOCUPADO. No se valida aquí cada variable de ingresos, remesas o vivienda ni se certifica un panel longitudinal. El control externo de 2025-T4 procede del [BCRD, PERN 2026-I, p. 23](https://cdn.bancentral.gov.do/documents/publicaciones-economicas/panorama-economico-de-la-region-norte/documents/PERN_2026-I.pdf).

Para repetir la verificación en vivo: ejecutar `verify-encft.R` mediante `tools/run-r.ps1`, con `PG_RECOVERY_PASSWORD` únicamente en el entorno, y reconstruir este cuaderno. Consulte `README.md` para el orden completo de reproducción.